# Lab 8: Introduction au Framework ADK et Multi-Provider

**Navigation** : [Index](../../README.md) | [Précédent <<](../../Track1-LangChain/Day3-Data-Agents/Labs/Lab7-Data-Analysis-Agent/Lab7-Data-Analysis-Agent.ipynb) | [Suivant >>](Lab9-First-ADK-Agent.ipynb)

## Objectifs d'apprentissage

A la fin de ce laboratoire, vous saurez :
1. Expliquer l'architecture du Google Agent Development Kit (ADK)
2. Configurer un environnement multi-provider (Gemini, vLLM, OpenAI)
3. Créer un premier client LLM avec votre provider choisi
4. Comparer les réponses de différents providers sur le même prompt

### Prérequis
- Python 3.10+
- Fichier `.env` configuré avec au moins un provider (voir section Configuration)
- Connaissance de base des LLMs (API, tokens, temperature)

### Durée estimée : 45-60 minutes

***

## Configuration requise

Avant de commencer, assurez-vous d'avoir configuré votre fichier `.env` :

```bash
# Exemple de configuration .env
ACTIVE_PROVIDER=gemini  # ou vllm, openai

# Gemini (optionnel)
GOOGLE_API_KEY=your_gemini_key

# vLLM (optionnel)
VLLM_BASE_URL=https://your-vllm-endpoint.com/v1
VLLM_MODEL=Qwen/Qwen2.5-72B-Instruct

# OpenAI (optionnel)
OPENAI_API_KEY=sk-...
```

***

## 1. Architecture Google ADK

Le **Agent Development Kit (ADK)** de Google est un framework pour construire des agents IA avec :

> **Repères bibliographiques.** Le concept d'agent IA à base de LLM (LLM-as-agent : perception → raisonnement → action via *tools*) est formalisé dans la synthèse de référence Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2023. Le framework **Google ADK** (Agent Development Kit) en fournit une implémentation officielle (`google.github.io/adk-docs`, dépôt `google/adk-python`), et **LangChain** — lancé en octobre 2022 par H. Chase — popularise l'approche modulaire (composabilité de chaînes, *tools*, mémoire) sur laquelle s'appuie une large part de l'écosystème agent Python.

- **Agents** : Entités qui interagissent via des tools
- **Tools** : Fonctions que l'agent peut appeler
- **Sessions** : Gestion de l'état conversationnel
- **Memory** : Persistance du contexte

### Comparaison avec LangChain

| Aspect | ADK | LangChain |
|--------|-----|----------|
| **Philosophie** | Google-first, intégré GCP | Multi-provider natif |
| **Agents** | Agent classes avec tools | Runnable, chains, agents |
| **Mémoire** | Session state intégrée | Modules séparés |
| **Déploiement** | Vertex AI Agent Engine | Variable |
| **Providers** | Gemini natif, OpenAI compatible | 50+ providers |

## 2. Configuration de l'Environnement

### Installation des dépendances

In [1]:
# Installation des dependances (decommenter si necessaire)
# !pip install google-adk google-genai litellm pydantic-settings

Import des modules et verification de l'installation.

In [2]:
import sys
import warnings
import os
from pathlib import Path

# Chemin vers Track2-GoogleADK depuis le répertoire de travail
track_dir = Path(os.getcwd()) / 'MyIA.AI.Notebooks' / 'ML' / 'DataScienceWithAgents' / 'Track2-GoogleADK'
sys.path.insert(0, str(track_dir))

warnings.filterwarnings('ignore', message=r'.*EXPERIMENTAL.*', category=UserWarning, module=r'google.adk')

from config import get_settings, get_provider_config
from utils.adk_runtime import build_agent, run_agent_turn

print('Modules importes avec succes')

Modules importes avec succes


### Vérification de la configuration

In [3]:
settings = get_settings()
config = get_provider_config(settings)

print(f'Provider actif: {config.provider.value}')
print(f'Modele: {config.model}')
print(f'Base URL: {config.base_url}')
api_display = '***' if config.api_key else 'Non requis (local)'
print(f'Cle API: {api_display}')

Provider actif: openrouter
Modele: openai/gpt-4.1-mini
Base URL: https://openrouter.ai/api/v1
Cle API: ***


Configuration detaillee du provider LLM.

In [4]:
print('Configuration complete :')
print(f'  Provider: {config.provider.value}')
print(f'  Model: {config.model}')
print(f'  API Key: {"Set" if config.api_key else "Not required"}')
print(f'  Base URL: {config.base_url}')

Configuration complete :
  Provider: openrouter
  Model: openai/gpt-4.1-mini
  API Key: Set
  Base URL: https://openrouter.ai/api/v1


## 3. Premier Test avec le Client LLM

Utilisons notre couche d'abstraction pour envoyer un prompt simple.

In [5]:
agent_intro = build_agent(
    name='lab8_intro',
    description='Agent d introduction ADK',
    instruction='Assistant expert ADK, reponds de maniere concise et precise.',
    config=config
)

response = await run_agent_turn(agent_intro, 'Explique en 2 phrases l architecture ADK.')

print('Reponse :')
print(response.response_text)

Reponse :
L’architecture ADK est une structure modulaire permettant de développer des applications en intégrant des composants réutilisables et interconnectés. Elle facilite la maintenance et l’extensibilité en séparant clairement les couches fonctionnelles et techniques.


Test de generation simple avec le client configure.

In [6]:
response2 = await run_agent_turn(
    agent_intro,
    'Quels sont les 3 composants principaux d un agent ADK ?'
)

print('Composants ADK :')
print(response2.response_text)

Composants ADK :
Les 3 composants principaux d’un agent ADK sont :

1. **Capteurs** – pour percevoir l’environnement.  
2. **Processeur** – pour traiter les informations et prendre des décisions.  
3. **Actionneurs** – pour agir sur l’environnement.


### Test avec prompt système

In [7]:
agent_system = build_agent(
    name='lab8_system',
    description='Agent avec instruction systeme',
    instruction='Tu es un expert en IA. Explique les concepts de maniere claire.',
    config=config
)

response3 = await run_agent_turn(
    agent_system,
    'Explique le concept de Session dans ADK.'
)

print('Session dans ADK :')
print(response3.response_text)

Session dans ADK :
Bien sûr ! Voici une explication claire du concept de **Session** dans l’**ADK** (Agent Development Kit) :

---

### Qu’est-ce qu’une **Session** dans l’ADK ?

Une **Session** dans l’ADK est une instance de communication ou d’interaction entre un utilisateur (ou un client) et un agent IA. Elle permet de suivre et gérer tout ce qui se passe durante une série d’échanges, depuis le début jusqu’à la fin de la conversation.

---

### Pourquoi utiliser une Session ?

- **Conserver le contexte**  
  La Session garde en mémoire les données et le contexte échangés au cours de la conversation. Cela permet à l’agent de comprendre et de répondre en tenant compte des informations précédentes.

- **Persistance temporaire des données**  
  Les informations pertinentes (comme les préférences utilisateur, les réponses précédentes, les variables du dialogue) sont stockées pendant toute la durée de la session.

- **Gestion personnalisée**  
  Une session unique permet de suivre l’état 

## 4. Comparaison Multi-Provider

Testons le même prompt avec différents providers pour comparer les réponses.

In [8]:
from config.providers import ProviderType

print('Providers disponibles:')
for p in ProviderType:
    print(f'  - {p.value}')

r_mp = await run_agent_turn(agent_intro, 'Quelle est la difference entre un LLM et un agent ?')
print('Reponse:', r_mp.response_text[:200], '...')

Providers disponibles:
  - gemini
  - openai
  - openrouter
  - vllm
  - lmstudio


Reponse: Un **LLM (Large Language Model)** est un modèle d'IA spécialisé dans le traitement et la génération de texte basé sur de grandes quantités de données.

Un **agent** utilise souvent un LLM, mais inclut ...


### Interprétation

La réponse ci-dessus provient du provider configuré dans `.env` (`ACTIVE_PROVIDER`).

Pour tester un autre provider, modifiez votre fichier `.env` et redémarrez le kernel, ou instanciez un client avec une configuration explicite :

```python
from config import ProviderConfig, ProviderType

# Exemple pour vLLM
vllm_config = ProviderConfig(
    provider=ProviderType.VLLM,
    model="Qwen/Qwen2.5-72B-Instruct",
    base_url="https://your-vllm-endpoint.com/v1",
    api_key=None
)
vllm_client = LLMClient(vllm_config)
```

## 5. Interface de Chat avec Historique

Le client supporte également une interface de chat avec historique des messages.

In [9]:
sid = 'lab8_chat_session'

r_a = await run_agent_turn(agent_intro, 'Bonjour, je suis nouveau avec ADK.', session_id=sid)
r_b = await run_agent_turn(agent_intro, 'Peux-tu me rappeler ce que j ai dit ?', session_id=sid)
print('Tour 1:', r_a.response_text)
print('Tour 2:', r_b.response_text)

Tour 1: Bonjour ! Bienvenue avec ADK. Comment puis-je vous aider à démarrer ?
Tour 2: Tu m'as demandé : « Peux-tu me rappeler ce que j ai dit ? »


## 6. Architecture des Frameworks DS-STAR et MLE-STAR

DS-STAR et MLE-STAR sont deux agents de référence (State-of-the-Art) conçus par Google Research pour la data science et l'ingénierie ML — ils incarnent le paradigme d'agent LLM *planner-coder* (décomposition de tâche → génération de code → exécution → raffinement) décrit par Xi et al. (2025) et appliqué à la compétition Kaggle et au cycle d'expérimentation ML.

Ce track Track2-GoogleADK intègre les frameworks de recherche Google :

### DS-STAR (Data Science Agent)

Architecture Planner-Coder-Verifier pour la data science autonome :

```mermaid
flowchart TD
    FA["File Analyzer"] --> P["Planner"]
    P --> C["Coder"]
    P --> V["Verifier"]
    C --> E["Executor"]
    E --> V
```

Le planificateur distribue le travail entre génération et vérification, tandis que l'exécuteur renvoie les résultats au vérificateur pour fermer la boucle de raffinement.

**Performance** : 45.2% accuracy sur DABStep benchmark

### MLE-STAR (ML Engineering Agent)

Extension avec recherche web et optimisation automatique :

- Web Search pour modèles SOTA
- Ablation studies ciblées
- Ensemble stratégies automatisées

**Performance** : 63.6% médailles sur MLE-Bench-Lite

## Exercice : Comparaison Multi-Provider

Maintenant que vous avez compris l'architecture, testez votre capacité à utiliser différents providers pour la même tâche.

In [10]:
# Exercice : Comparaison Multi-Provider
# Objectif : Creer une fonction qui compare les reponses de differents agents/ Providers sur le meme prompt
# TODO: Definissez une liste de prompts de test pertinents pour la data science
# Indice : utilisez des questions sur l'ADK, les agents, ou la data science
test_prompts = None  # Exemple: ['Explique l'architecture ADK.', 'A quoi sert un agent ?']

# TODO: Implementez une fonction qui execute chaque prompt sur plusieurs agents/providers
# Utilisez build_agent pour creer des agents avec differentes configurations
# puis run_agent_turn pour executer chaque prompt
# Indice: mesurez le temps avec time.time()
async def compare_providers(prompts, config):
    """
    Compare les reponses de differents agents pour une liste de prompts.
        Args:
        prompts: Liste de prompts a tester
        config: Configuration ADK
            Returns:
        dict: Resultats par prompt avec reponses et temps
    """
    results = {}
    # TODO: Implementez la boucle de test
    # Indice: creez des agents avec build_agent, puis appelez run_agent_turn
    # for prompt in prompts:
    #     agent = build_agent(name='test', instruction='Reponds de maniere concise.', config=config)
    #     start = time.time()
    #     resp = await run_agent_turn(agent, prompt)
    #     results[prompt] = {'response': resp.response_text, 'time': time.time() - start}
    return results

# TODO: Executez la comparaison et affichez les resultats
# results = await compare_providers(test_prompts, config)
# for prompt, data in results.items():
#     print(f"Prompt: {prompt[:50]}...")
#     print(f"Reponse: {data['response'][:100]}...")
#     print(f"Temps: {data['time']:.2f}s")
print("Exercice a completer : comparaison multi-provider avec ADK")

Exercice a completer : comparaison multi-provider avec ADK


## Exercice : Chat Multi-Tours avec Contexte Data Science

Utilisez l'interface `chat()` du client LLM pour construire une conversation specialisee en data science. L'objectif est de simuler un assistant qui guide un utilisateur dans son analyse de données étapes par étapes.

### Objectifs
1. Construire un historique de conversation avec 3 echanges
2. Utiliser un system prompt specialise data science
3. Observer comment le contexte précédent influence les reponses

**Indice :**
- Utilisez `client.chat(messages)` avec une liste de dictionnaires `{"rôle": "user"/"assistant", "content": "..."}`
- Commencez par un message system via le premier élément de la liste

In [11]:
# Exercice : Chat Multi-Tours avec Contexte Data Science
# Objectif : Simuler un assistant d'analyse de donnees en 3 echanges
# TODO: Construisez un agent specialise en data science
# Utilisez build_agent avec une instruction appropriee
ds_agent = None  # Exemple: build_agent(name='ds_expert', instruction='Tu es un expert en data science.', config=config)

# TODO: Definissez un identifiant de session pour maintenir le contexte
ds_sid = None  # Exemple: 'ds_session_001'

# TODO: Premier echange - l'utilisateur decrit son dataset
# Indice: utilisez run_agent_turn avec session_id pour maintenir l'historique
# question_1 = "J'ai un dataset de ventes avec date, produit, region, quantite et prix."
# resp_1 = await run_agent_turn(ds_agent, question_1, session_id=ds_sid)

# TODO: Deuxieme echange - l'utilisateur demande une analyse specifique
# question_2 = "Quelles visualisations me recommandes-tu pour comparer les regions ?"
# resp_2 = await run_agent_turn(ds_agent, question_2, session_id=ds_sid)

# TODO: Troisieme echange - approfondissement
# question_3 = "Comment detecter des valeurs aberrantes dans les prix ?"
# resp_3 = await run_agent_turn(ds_agent, question_3, session_id=ds_sid)

# TODO: Affichez les 3 reponses et observez la progression du contexte
# print("=== Echange 1 ===")
# print(resp_1.response_text)
# print("=== Echange 2 ===")
# print(resp_2.response_text)
# print("=== Echange 3 ===")
# print(resp_3.response_text)
print("Exercice a completer : chat multi-tours avec agent data science via ADK")

Exercice a completer : chat multi-tours avec agent data science via ADK


## Exercice : Exploration des Paramètres de Generation

Experimentez avec les paramètres `temperature` et `max_tokens` pour comprendre leur impact sur la qualite des reponses d'un agent. L'objectif est de trouver les paramètres optimaux pour différentes tâches d'agent.

### Objectifs
1. Tester 3 valeurs de temperature (0.1, 0.7, 1.5) sur un prompt technique
2. Observer l'impact de `max_tokens` sur la longueur des reponses
3. Determiner les paramètres ideaux pour du code generation vs. du texte creatif

**Indice :**
- `client.generate(prompt, temperature=0.1, max_tokens=100)` pour contrôler la generation
- Temperature basse = reponses déterministes (ideal pour du code)
- Temperature haute = reponses creatives (ideal pour du brainstorming)

In [12]:
# Exercice : Exploration des parametres de generation
# Objectif : Trouver les parametres optimaux selon la tache de l'agent
# TODO: Definissez des prompts pour differentes taches
prompt_technique = "Explique la difference entre un agent base sur des tools et un agent base sur du code execution."  # Ne pas modifier
prompt_code = "Ecris une fonction Python qui calcule la moyenne mobile d'une serie temporelle."  # Ne pas modifier

# TODO: Testez differentes temperatures et comparez les reponses
# Utilisez build_agent avec differentes configurations de temperature
# puis run_agent_turn pour executer les prompts
# Indice: essayez temperature=0.1, 0.7, 1.5
temperatures = [0.1, 0.7, 1.5]
results_temp = {}

# TODO: Implementez la boucle de test pour les temperatures
# for t in temperatures:
#     agent = build_agent(name='temp_test', instruction='Reponds de maniere detaillee.', config=config, temperature=t)
#     resp = await run_agent_turn(agent, prompt_technique)
#     results_temp[t] = resp.response_text
#     print(f"--- Temperature {t} ---")
#     print(results_temp[t][:200])
#     print()

# TODO: Testez l'impact de max_tokens sur un prompt de code
# Indice: essayez max_tokens=50 vs max_tokens=500
# agent_short = build_agent(name='short', instruction='Ecris du code concis.', config=config, max_tokens=50)
# agent_long = build_agent(name='long', instruction='Ecris du code complet.', config=config, max_tokens=500)
# resp_short = await run_agent_turn(agent_short, prompt_code)
# resp_long = await run_agent_turn(agent_long, prompt_code)
# print(f"Short (50 tokens): {resp_short.response_text[:100]}...")
# print(f"Long (500 tokens): {resp_long.response_text[:200]}...")

# TODO: Resumez vos conclusions dans un dictionnaire
# Adaptez les valeurs en fonction de vos observations
parametres_ideaux = {
    "code_generation": {"temperature": None, "max_tokens": None},  # Remplacez None
    "analyse_creative": {"temperature": None, "max_tokens": None},  # Remplacez None
    "brainstorming": {"temperature": None, "max_tokens": None}   # Remplacez None
}
print("Exercice a completer : exploration des parametres temperature et max_tokens avec ADK")

Exercice a completer : exploration des parametres temperature et max_tokens avec ADK


## Résumé et Prochaines Étapes

### Ce que nous avons appris

1. **Configuration multi-provider** : Un seul fichier `.env` permet de switcher entre Gemini, vLLM, OpenAI
2. **Abstraction LiteLLM** : Interface unifiée pour tous les providers
3. **Client LLM simple** : `generate()` et `chat()` pour interagir avec n'importe quel modèle
4. **Architecture DS-STAR** : Framework Planner-Coder-Verifier pour la data science autonome

### Points clés à retenir

| Concept | Description |
|---------|-------------|
| `ProviderConfig` | Configuration d'un provider LLM |
| `LLMClient` | Client unifié pour tous les providers |
| `generate()` | Génération simple avec prompt |
| `chat()` | Conversation multi-tours avec historique |

### Prochaines étapes

- **Lab 9** : Créer un premier agent ADK avec tools Python pour analyser des DataFrames
- **Lab 10** : Implémenter le File Analyzer de DS-STAR
- **Lab 11** : Boucle Planner-Coder-Verifier

***

**Navigation** : [Index](../../README.md) | [Précédent <<](../../Track1-LangChain/Day3-Data-Agents/Labs/Lab7-Data-Analysis-Agent/Lab7-Data-Analysis-Agent.ipynb) | [Suivant >> Lab 9 - First ADK Agent](Lab9-First-ADK-Agent.ipynb)

## Ressources

- [Google ADK Documentation](https://github.com/google/adk-samples)
- [DS-STAR Paper](https://research.google/blog/ds-star-a-state-of-the-art-versatile-data-science-agent/)
- [MLE-STAR Paper](https://research.google/blog/mle-star-a-state-of-the-art-machine-learning-engineering-agents/)
- [LiteLLM Documentation](https://docs.litellm.ai/)

## Références

1. Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2023. Synthèse de référence sur les agents IA à base de LLM (architecture perception-raisonnement-action, *tools*, mémoire, cadres planner-coder).
2. Google, *Agent Development Kit (ADK)*, documentation officielle, `google.github.io/adk-docs` (dépôt `google/adk-python`). Framework agent-first de Google (sessions, mémoire, Vertex AI Agent Engine).
3. H. Chase, *LangChain*, octobre 2022, `langchain.com` / `github.com/langchain-ai/langchain`. Framework modulaire open-source pour applications LLM (chaînes composables, *tools*, mémoire) — référence de l'écosystème agent Python.
4. Google Research, *DS-STAR: A State-of-the-Art Versatile Data Science Agent*, 2025, `research.google/blog/ds-star-a-state-of-the-art-versatile-data-science-agent/`. Agent SOTA data science (paradigme planner-coder).
5. Google Research, *MLE-STAR: A State-of-the-Art Machine Learning Engineering Agent*, 2025, `research.google/blog/mle-star-a-state-of-the-art-machine-learning-engineering-agents/`. Agent SOTA ingénierie ML (cycle d'expérimentation, Kaggle).